In [ ]:
import tmap as tm
from faerun import Faerun
from mhfp.encoder import MHFPEncoder
import numpy as np
import pandas as pd
import pubchempy as pcp
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold
from matplotlib import colors as mcolors

In [ ]:
def add_scaffold_smiles(dataframe, smiles_column):
    """
    根据 SMILES 列生成其对应的 scaffold_smiles 列。
    
    参数:
    - dataframe: 包含 SMILES 数据的 pandas DataFrame
    - smiles_column: SMILES 数据所在列的列名 (字符串)
    
    返回:
    - 带有 scaffold_smiles 列的新 DataFrame
    """
    scaffold_smiles = []
    for smiles in dataframe[smiles_column]:
        mol = Chem.MolFromSmiles(smiles)
        if mol is not None:  # 检查 SMILES 是否能够被正确解析
            scaffold = MurckoScaffold.GetScaffoldForMol(mol)
            scaffold_smiles.append(Chem.MolToSmiles(scaffold))
        else:
            scaffold_smiles.append(None)  # 如果解析失败，填入 None

    # 添加新列到 DataFrame
    dataframe['murckoscaffold'] = scaffold_smiles
    return dataframe

In [ ]:
gen_mol = pd.read_csv("./generated_molecules.csv")

In [ ]:
gen_mol = gen_mol.drop_duplicates(keep='first', subset='smiles').reset_index(drop=True)

In [ ]:
#按骨架核心分类
df_copy = gen_mol.copy()

df_1 = pd.DataFrame(columns=df_copy.columns)
df_2 = pd.DataFrame(columns=df_copy.columns)
df_3 = pd.DataFrame(columns=df_copy.columns)
df_4 = pd.DataFrame(columns=df_copy.columns)
df_5 = pd.DataFrame(columns=df_copy.columns)
df_6 = pd.DataFrame(columns=df_copy.columns)
df_7 = pd.DataFrame(columns=df_copy.columns)

# 链状
for index, row in df_copy.iterrows():
    smiles = row['smiles']
    mol = Chem.MolFromSmiles(smiles)
    if mol is not None:
        ring_info = mol.GetRingInfo()
        num_rings = ring_info.NumRings()
        if num_rings == 0:
            df_1 = pd.concat([df_1, pd.DataFrame([row])], ignore_index=True)
            df_copy.drop(index, inplace=True)

# 萘酰胺
patt_1 = Chem.MolFromSmiles('O=C1NC(=O)C2=CC=CC3=C2C1=CC=C3')
for index, row in df_copy.iterrows():
    smiles = row['smiles']
    mol = Chem.MolFromSmiles(smiles)
    if mol.HasSubstructMatch(patt_1):
        df_2 = pd.concat([df_2, pd.DataFrame([row])], ignore_index=True)
        df_copy.drop(index, inplace=True)

# 三苯胺
patt_2 = Chem.MolFromSmiles('C1=CC=C(C=C1)N(C1=CC=CC=C1)C1=CC=CC=C1')
for index, row in df_copy.iterrows():
    smiles = row['smiles']
    mol = Chem.MolFromSmiles(smiles)
    if mol.HasSubstructMatch(patt_2):
        df_3 = pd.concat([df_3, pd.DataFrame([row])], ignore_index=True)
        df_copy.drop(index, inplace=True)

# 咔唑
patt_3 = Chem.MolFromSmiles('N1C2=C(C=CC=C2)C2=C1C=CC=C2')
for index, row in df_copy.iterrows():
    smiles = row['smiles']
    mol = Chem.MolFromSmiles(smiles)
    if mol.HasSubstructMatch(patt_3):
        df_4 = pd.concat([df_4, pd.DataFrame([row])], ignore_index=True)
        df_copy.drop(index, inplace=True)

# 联噻吩
patt_4 = Chem.MolFromSmiles('S1C=CC=C1C1=CC=CS1')
for index, row in df_copy.iterrows():
    smiles = row['smiles']
    mol = Chem.MolFromSmiles(smiles)
    if mol.HasSubstructMatch(patt_4):
        df_5 = pd.concat([df_5, pd.DataFrame([row])], ignore_index=True)
        df_copy.drop(index, inplace=True)

# 单环
for index, row in df_copy.iterrows():
    smiles = row['smiles']
    mol = Chem.MolFromSmiles(smiles)
    if mol is not None:
        ring_info = mol.GetRingInfo()
        num_rings = ring_info.NumRings()
        if num_rings == 1:
            df_6 = pd.concat([df_6, pd.DataFrame([row])], ignore_index=True)
            df_copy.drop(index, inplace=True)

# 其他
df_7 = df_copy.copy()

In [ ]:
print(df_1.shape)
print(df_2.shape)
print(df_3.shape)
print(df_4.shape)
print(df_5.shape)
print(df_6.shape)
print(df_7.shape)

In [ ]:
sam_mcs_matching = pd.concat([df_2, df_3, df_4, df_5, df_6], axis=0).reset_index(drop=True)
sam_mcs_matching

In [ ]:
gen_smiles = sam_mcs_matching
gen_smiles = add_scaffold_smiles(gen_smiles, 'smiles')

In [ ]:
sca_condition = pd.read_csv("./scaffold_for_generation.csv").iloc[:,0:1]
sca_condition.head()

In [ ]:
sam4tl = pd.read_csv("./reported_sam_with_length_100.csv").iloc[:,0:1]
sam4tl

In [ ]:
sam4tl["label"]=2

In [ ]:
sca_input_set = set(sca_condition['scaffold_smiles'])
# 添加 label 列
gen_smiles['label'] = gen_smiles['murckoscaffold'].apply(
    lambda x: 0 if x in sca_input_set else 1
)

In [ ]:
gen_smiles

In [ ]:
# 统计标签的数量
label_counts = gen_smiles['label'].value_counts()

# 打印结果
print("标签统计：")
print(f"标签为 0 的数量: {label_counts.get(0, 0)}")
print(f"标签为 1 的数量: {label_counts.get(1, 0)}")

In [ ]:
sam_merged = pd.concat([gen_smiles, sam4tl], axis=0).reset_index(drop=True)

In [ ]:
sam_merged

In [ ]:
smiles = sam_merged['smiles']
labels = sam_merged['label']

In [ ]:
perm = 512
enc = MHFPEncoder(perm)
fingerprints = [tm.VectorUint(enc.encode(s)) for s in sam_merged['smiles']]
fingerprints_array = np.array(fingerprints)
print("Fingerprints 的形状:", fingerprints_array.shape)

In [ ]:
# Initialize the LSH Forest
lf = tm.LSHForest(perm)
 
# Add the Fingerprints to the LSH Forest and index
lf.batch_add(fingerprints)
lf.index()

In [ ]:
cfg = tm.LayoutConfiguration()
cfg.k = 100
cfg.sl_repeats = 2
cfg.mmm_repeats = 2
cfg.node_size = 2
x, y, s, t, _ = tm.layout_from_lsh_forest(lf, config=cfg)

In [ ]:
label_colors = {
    0: "#DAA628",
    1: "#719AAC",
    2: "#C74D26"
}

# 2) 更新 cmap 函数以适应新标签
def cmap(val):
    """Faerun 会把标签值（float/int）传进来；返回 RGBA list"""
    return mcolors.to_rgba(label_colors.get(int(val), "#000000"))  # 未知标签→黑色

# 3) 图例文本
legend_labels = [(k, f"Label {k}") for k in sorted(label_colors)]

# -------------------------------------------------
# 4) 创建 Faerun，并绘制散点图
f = Faerun(clear_color="#ffffff", coords=False, view="front",
           impress="Made with Faerun and tmap")

# 添加可点击的散点
f.add_scatter(
    "Molecules",
    {
        "x": x,
        "y": y,
        "c": labels,                               # 直接用 0-2
        "labels": [f"{smiles[i]} (Label: {labels[i]})"
                   for i in range(len(smiles))],
    },
    colormap=cmap,                                # 传入 callable
    categorical=[True],
    has_legend=True,
    legend_labels=legend_labels,
    shader="smoothCircle",
    point_scale=5,
)

# 2) 添加树边（图层连接）
f.add_tree(
    "Molecule_Tree",           # 图层名称自定义
    {
    "from": s, "to": t
    },
    point_helper="Molecules",  # 告诉 Faerun 用哪一层里的坐标
    color="#888888",           # 边的颜色/粗细可自调
)

# 5) 输出HTML，启用交互
f.plot("molecule_visualization_output", template="smiles")

In [ ]:
# 1. 预处理输入 SMILES，转为标准形式（canonical）
query_smiles = ['OP(O)(=O)OC1=CC=C(C=C1)N(C1=CC=CC=C1)C1=CC=C(C=C1)C1=CC=CC2=NSN=C12']

def safe_parse(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is not None:
            return Chem.MolToSmiles(mol)
    except:
        return None

query_canonical = list(filter(None, [safe_parse(s) for s in query_smiles]))

# 2. 用临时 canonical_smiles 不修改 gen_smiles 本身
def canonical_match(row_smiles):
    try:
        mol = Chem.MolFromSmiles(row_smiles)
        if mol is not None:
            return Chem.MolToSmiles(mol)
    except:
        return None

# 3. 找出匹配行索引
matched_idx = gen_smiles.index[
    gen_smiles['smiles'].apply(canonical_match).isin(query_canonical)
]

# 4. 返回原始列
matched_rows = gen_smiles.loc[matched_idx, gen_smiles.columns]

In [ ]:
matched_rows

In [ ]:
from rdkit.Chem import Draw
from IPython.display import display

# 取匹配行第一条
row = matched_rows.iloc[0]

# 提取 smiles 和 scaffold_smiles
smiles_1 = row['smiles']
smiles_2 = row['scaffold_smiles']

# 转为分子对象
mol_1 = Chem.MolFromSmiles(smiles_1)
mol_2 = Chem.MolFromSmiles(smiles_2)

# 画图
img = Draw.MolsToGridImage([mol_1, mol_2],
                           molsPerRow=2,
                           subImgSize=(300, 300),
                           legends=["smiles", "scaffold_smiles"])

# 显示图像
display(img)